> **Public snapshot note**
> 이 노트북은 원래 연구 실행 흐름을 보존한 archival artifact입니다.
> 공개 스냅샷에서는 raw PDF, processed corpus, `kg_gen/graphrag_workspace/input`, `kg_gen/graphrag_workspace/output`,
> `kg_gen/snu_kg_output/graphrag_workspace/output*` 등 bulk/source-derived 자산을 의도적으로 제외했습니다.
> 따라서 아래의 일부 경로 설명은 **원래 실험 환경 기준**이며, 현재 public snapshot과 1:1로 대응하지 않을 수 있습니다.


# GraphRAG를 이용한 한국 농업 지식그래프 생성 (공식 문서 기반)

이 노트북은 Microsoft GraphRAG 공식 문서의 권장 방식을 따라 한국 농업 데이터에서 지식그래프를 생성합니다.

## 📚 주요 특징:
- **공식 파이프라인 준수**: GraphRAG 문서의 Best Practice 적용
- **워크스페이스 분리**: 소스코드와 작업공간을 명확히 분리
- **한국 농업 특화**: 도메인 적응형 프롬프트 자동 생성
- **강화된 에러 처리**: API 오류, 타임아웃 등 실무 환경 고려

## 🔧 GraphRAG 작동 원리:
1. **문서 청킹**: 입력 문서를 TextUnits으로 분할
2. **엔티티/관계 추출**: LLM을 통한 지식 추출
3. **계층적 클러스터링**: Leiden 알고리즘으로 커뮤니티 구조 생성
4. **커뮤니티 요약**: 각 커뮤니티의 특성 요약
5. **쿼리 지원**: Local/Global/DRIFT Search 제공

## ⚠️ 실행 전 필수 사항:
1. **OpenAI API 키**: 유효한 API 키와 충분한 크레딧 필요
2. **Python 환경**: Python 3.10 이상
3. **디스크 공간**: 최소 1GB 여유 공간

## 0. 환경 설정 및 GraphRAG 설치

In [ ]:
# 필수 라이브러리 임포트
import json
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import yaml
from datetime import datetime
import time
import threading

# 프로젝트 경로 설정
PROJECT_ROOT = "<PROJECT_ROOT>"
GRAPHRAG_ROOT = os.path.join(PROJECT_ROOT, "kg_gen/graphrag")
DATA_PATH = os.path.join(PROJECT_ROOT, "rag_dataset/processed_texts/validated_texts_all_20251119_015554.json")
WORK_DIR = os.path.join(PROJECT_ROOT, "kg_gen/graphrag_workspace")

print("📁 프로젝트 설정:")
print(f"   프로젝트 루트: {PROJECT_ROOT}")
print(f"   GraphRAG 소스: {GRAPHRAG_ROOT}")
print(f"   작업 디렉토리: {WORK_DIR}")
print(f"   데이터 경로: {DATA_PATH}")

In [ ]:
# GraphRAG 설치 확인 및 설정
print("🔍 GraphRAG 설치 상태 확인...")

# 1. GraphRAG 모듈 확인
try:
    import graphrag
    print("✅ GraphRAG 모듈이 설치되어 있습니다.")
except ImportError:
    print("❌ GraphRAG가 설치되지 않았습니다.")
    print(f"\n💡 설치 방법:")
    print(f"cd {GRAPHRAG_ROOT}")
    print(f"pip install -e .")
    raise SystemExit("GraphRAG 설치 후 다시 실행하세요.")

# 2. GraphRAG CLI 확인
try:
    result = subprocess.run(
        [sys.executable, "-m", "graphrag", "--help"],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        print("✅ GraphRAG CLI 명령어 사용 가능")
    else:
        print("⚠️ GraphRAG CLI 실행 문제 발생")
except Exception as e:
    print(f"❌ GraphRAG CLI 테스트 실패: {e}")

print("\n" + "="*50)

## 1. API 키 설정 및 검증

In [ ]:
# OpenAI API 키 설정
import getpass

# API 키 입력 (보안을 위해 getpass 사용)
if "OPENAI_API_KEY" not in os.environ:
    print("🔑 OpenAI API 키를 입력하세요:")
    api_key = getpass.getpass("API Key: ")
    os.environ["OPENAI_API_KEY"] = api_key
    os.environ["GRAPHRAG_API_KEY"] = api_key
else:
    print("✅ API 키가 이미 설정되어 있습니다.")
    api_key = os.environ["OPENAI_API_KEY"]
    os.environ["GRAPHRAG_API_KEY"] = api_key

# API 키 마스킹 표시
masked_key = api_key[:8] + "..." + api_key[-4:] if len(api_key) > 12 else "***"
print(f"🔐 설정된 API 키: {masked_key}")

In [ ]:
# OpenAI API 연결 테스트
print("🧪 OpenAI API 연결 테스트...")

try:
    from openai import OpenAI
    
    client = OpenAI(api_key=api_key)
    
    # 간단한 테스트 요청
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "한국 농업에서 '키토산'의 역할을 한 줄로 설명하세요."}
        ],
        max_tokens=50,
        temperature=0.1
    )
    
    print("✅ API 연결 성공!")
    print(f"📝 테스트 응답: {response.choices[0].message.content}")
    
    # GraphRAG가 사용할 모델 테스트
    models_to_test = ["gpt-4o", "text-embedding-ada-002"]
    
    for model in models_to_test:
        try:
            if "embedding" in model:
                # 임베딩 모델 테스트
                response = client.embeddings.create(
                    model=model,
                    input="테스트 텍스트"
                )
                print(f"✅ {model} 모델 접근 가능")
            else:
                # 채팅 모델 테스트
                response = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": "test"}],
                    max_tokens=5
                )
                print(f"✅ {model} 모델 접근 가능")
        except Exception as e:
            print(f"⚠️ {model} 모델 접근 실패: {str(e)[:50]}...")
            
except Exception as e:
    print(f"❌ API 테스트 실패: {e}")
    print("\n🔧 해결 방법:")
    print("1. API 키가 올바른지 확인")
    print("2. OpenAI 계정에 크레딧이 있는지 확인")
    print("3. 네트워크 연결 상태 확인")

## 2. 데이터 로드 및 전처리

In [ ]:
# 검증된 농업 데이터 로드
print("📊 농업 데이터 로드 중...")

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    validated_data = json.load(f)

print(f"\n✅ 총 {len(validated_data)}개의 문서 로드됨")
print("\n📋 문서 정보:")

total_chars = 0
for i, doc in enumerate(validated_data[:5]):  # 처음 5개만 표시
    text_content = doc.get('text', '')
    metadata = doc.get('metadata', {})
    validation = doc.get('validation', {})
    
    # 문서 정보 추출
    title = metadata.get('filename', metadata.get('url', f'문서 {i+1}'))
    source_type = metadata.get('method', 'Unknown')
    text_len = len(text_content)
    total_chars += text_len
    
    # 농업 키워드 추출 (있는 경우)
    llm_response = validation.get('llm_response', '')
    keywords = []
    if '농업 키워드:' in llm_response:
        keyword_line = llm_response.split('농업 키워드:')[1].split('\n')[0]
        keywords = [k.strip() for k in keyword_line.split(',') if k.strip()]
    
    print(f"\n📄 문서 {i+1}:")
    print(f"   제목: {title[:60]}..." if len(title) > 60 else f"   제목: {title}")
    print(f"   소스: {source_type}")
    print(f"   크기: {text_len:,} 문자")
    print(f"   농업 관련: {'✅' if validation.get('is_agricultural', False) else '❌'}")
    if keywords:
        print(f"   키워드: {', '.join(keywords[:5])}" + ("..." if len(keywords) > 5 else ""))

if len(validated_data) > 5:
    print(f"\n... 외 {len(validated_data) - 5}개 문서")

print(f"\n📊 전체 통계:")
print(f"   총 문서 수: {len(validated_data)}개")
print(f"   총 문자 수: {total_chars:,}자")
print(f"   평균 문서 길이: {total_chars // len(validated_data):,}자")

In [ ]:
# 작업 디렉토리 생성 및 데이터 준비
print(f"🏗️ 작업 디렉토리 준비: {WORK_DIR}")

# 작업 디렉토리 생성
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# 입력 디렉토리 생성
input_dir = os.path.join(WORK_DIR, "input")
os.makedirs(input_dir, exist_ok=True)

print("\n📝 문서를 개별 텍스트 파일로 변환 중...")

# 각 문서를 개별 파일로 저장
file_info = []

for i, doc in enumerate(validated_data):
    # 파일명 생성
    filename = f"agriculture_doc_{i+1:03d}.txt"
    filepath = os.path.join(input_dir, filename)
    
    # 데이터 추출
    text_content = doc.get('text', '')
    metadata = doc.get('metadata', {})
    validation = doc.get('validation', {})
    
    # 메타데이터 정보
    title = metadata.get('filename', metadata.get('url', f'농업 문서 {i+1}'))
    source = metadata.get('method', 'Unknown')
    extracted_date = metadata.get('extracted_at', datetime.now().isoformat())
    
    # 농업 키워드 추출
    keywords = []
    llm_response = validation.get('llm_response', '')
    if '농업 키워드:' in llm_response:
        keyword_line = llm_response.split('농업 키워드:')[1].split('\n')[0]
        keywords = [k.strip() for k in keyword_line.split(',') if k.strip()]
    
    # 문서 내용 구성 (메타데이터 포함)
    content = f"""제목: {title}
출처: {source}
추출일: {extracted_date}
농업 관련: {'예' if validation.get('is_agricultural', False) else '아니오'}
{f'핵심 키워드: {', '.join(keywords)}' if keywords else ''}

{'='*80}

{text_content}"""
    
    # 파일 저장
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
    
    # 파일 정보 저장
    file_info.append({
        'filename': filename,
        'size': len(content),
        'title': title[:50] + '...' if len(title) > 50 else title,
        'keywords': keywords[:3]  # 상위 3개 키워드만
    })
    
    if (i + 1) % 10 == 0:
        print(f"   {i + 1}/{len(validated_data)} 파일 생성됨...")

print(f"\n✅ 총 {len(file_info)}개 파일 생성 완료!")

# 생성된 파일 요약
print("\n📊 생성된 파일 요약 (처음 5개):")
for info in file_info[:5]:
    print(f"   • {info['filename']}: {info['size']:,}자")
    print(f"     제목: {info['title']}")
    if info['keywords']:
        print(f"     키워드: {', '.join(info['keywords'])}")

if len(file_info) > 5:
    print(f"   ... 외 {len(file_info) - 5}개 파일")

# 전체 통계
total_size = sum(f['size'] for f in file_info)
print(f"\n📈 전체 통계:")
print(f"   총 파일 수: {len(file_info)}개")
print(f"   총 크기: {total_size:,}자")
print(f"   평균 파일 크기: {total_size // len(file_info):,}자")

## 3. GraphRAG 프로젝트 초기화

In [ ]:
# GraphRAG 프로젝트 초기화
print("🚀 GraphRAG 프로젝트 초기화...")
print(f"📁 작업 디렉토리: {WORK_DIR}")

# 기존 설정 파일 확인
settings_file = os.path.join(WORK_DIR, "settings.yaml")
prompts_dir = os.path.join(WORK_DIR, "prompts")

if os.path.exists(settings_file):
    print("\n⚠️ 기존 프로젝트가 발견되었습니다.")
    print("   기존 설정을 사용합니다.")
    print(f"   설정 파일: {settings_file}")
    print(f"   프롬프트 디렉토리: {prompts_dir}")
else:
    print("\n📦 새 프로젝트 초기화 중...")
    
    # graphrag init 실행
    init_cmd = [sys.executable, "-m", "graphrag", "init", "--root", "."]
    
    try:
        result = subprocess.run(
            init_cmd,
            cwd=WORK_DIR,
            capture_output=True,
            text=True,
            timeout=30
        )
        
        if result.returncode == 0:
            print("✅ 초기화 성공!")
        else:
            print(f"⚠️ 초기화 중 경고: {result.stderr}")
            
    except subprocess.TimeoutExpired:
        print("⏱️ 초기화 타임아웃 (30초)")
    except Exception as e:
        print(f"❌ 초기화 실패: {e}")

# 생성된 파일 확인
print("\n📋 프로젝트 구조 확인:")
for root, dirs, files in os.walk(WORK_DIR):
    level = root.replace(WORK_DIR, '').count(os.sep)
    indent = '  ' * level
    rel_path = os.path.relpath(root, WORK_DIR)
    
    if level < 3:  # 최대 3단계까지만 표시
        print(f"{indent}📁 {rel_path}/")
        
        subindent = '  ' * (level + 1)
        for file in files[:5]:  # 각 디렉토리당 5개 파일만
            if not file.startswith('.'):
                print(f"{subindent}📄 {file}")
        
        if len(files) > 5:
            print(f"{subindent}... ({len(files) - 5}개 파일 더)")

In [ ]:
# settings.yaml 수정 및 API 키 설정
print("⚙️ GraphRAG 설정 구성...")

# settings.yaml 로드
if os.path.exists(settings_file):
    with open(settings_file, 'r', encoding='utf-8') as f:
        settings = yaml.safe_load(f)
    
    print("📝 현재 설정을 수정합니다...")
    
    # 모델 설정 (models 섹션 사용)
    if 'models' not in settings:
        settings['models'] = {}
    
    # 기본 채팅 모델 설정
    settings['models']['default_chat_model'] = {
        'api_key': '${OPENAI_API_KEY}',
        'type': 'openai_chat',
        'model': 'gpt-4o',
        'model_supports_json': True,
        'max_tokens': 4000,
        'temperature': 0
    }
    
    # 기본 임베딩 모델 설정
    settings['models']['default_embedding_model'] = {
        'api_key': '${OPENAI_API_KEY}',
        'type': 'openai_embedding',
        'model': 'text-embedding-ada-002'
    }
    
    # 청킹 설정 (한국어 특성 고려)
    if 'chunks' not in settings:
        settings['chunks'] = {}
    
    settings['chunks']['size'] = 300  # 한국어는 토큰 효율이 낮으므로 작게 설정
    settings['chunks']['overlap'] = 100
    settings['chunks']['strategy'] = 'tokens'
    
    # 그래프 추출 설정
    if 'extract_graph' not in settings:
        settings['extract_graph'] = {}
    
    settings['extract_graph']['model_id'] = 'default_chat_model'
    settings['extract_graph']['max_gleanings'] = 1  # 비용 절감을 위해 1로 설정
    
    # 커뮤니티 리포트 설정
    if 'community_reports' not in settings:
        settings['community_reports'] = {}
    
    settings['community_reports']['model_id'] = 'default_chat_model'
    settings['community_reports']['max_length'] = 2000
    
    # 임베딩 설정
    if 'embed_text' not in settings:
        settings['embed_text'] = {}
    
    settings['embed_text']['model_id'] = 'default_embedding_model'
    settings['embed_text']['batch_size'] = 16
    
    # 설정 저장
    with open(settings_file, 'w', encoding='utf-8') as f:
        yaml.dump(settings, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
    
    print("✅ settings.yaml 업데이트 완료")
    
    # 주요 설정 표시
    print("\n📋 주요 설정:")
    print(f"   채팅 모델: {settings['models']['default_chat_model']['model']}")
    print(f"   임베딩 모델: {settings['models']['default_embedding_model']['model']}")
    print(f"   청크 크기: {settings['chunks']['size']} 토큰")
    print(f"   청크 오버랩: {settings['chunks']['overlap']} 토큰")
    
else:
    print("❌ settings.yaml 파일을 찾을 수 없습니다.")
    print("   GraphRAG 초기화를 먼저 실행하세요.")

In [ ]:
# .env 파일 생성 (환경 변수 설정)
print("🔐 환경 변수 파일 생성...")

env_file = os.path.join(WORK_DIR, ".env")
env_content = f"""# GraphRAG 환경 변수
OPENAI_API_KEY={api_key}
GRAPHRAG_API_KEY={api_key}

# 추가 설정 (선택사항)
# GRAPHRAG_LLM_MODEL=gpt-4o
# GRAPHRAG_EMBEDDING_MODEL=text-embedding-ada-002
"""

with open(env_file, 'w') as f:
    f.write(env_content)

print(f"✅ .env 파일 생성: {env_file}")
print(f"   API 키: {masked_key}")

# 환경 변수 로드
try:
    from dotenv import load_dotenv
    load_dotenv(env_file, override=True)
    print("✅ 환경 변수 로드 완료")
except ImportError:
    print("⚠️ python-dotenv가 없어도 GraphRAG는 .env 파일을 읽을 수 있습니다.")

## 4. Auto Prompt Tuning (한국 농업 도메인 특화)

In [ ]:
# 프롬프트 튜닝 설정
print("🎯 한국 농업 도메인 특화 프롬프트 튜닝 설정")
print("\n" + "="*60)

# 튜닝 파라미터 설정
TUNING_CONFIG = {
    'domain': 'Korean agriculture',  # 한국 농업 도메인
    'language': 'Korean',  # 한국어
    'selection_method': 'auto',  # 자동 선택 (대표 샘플)
    'limit': 15,  # 샘플 수
    'max_tokens': 2000,  # 최대 토큰
    'chunk_size': 300,  # 청크 크기
    'min_examples_required': 2,  # 최소 예제 수
    'discover_entity_types': True,  # 엔티티 타입 자동 발견
    'n_subset_max': 300,  # auto 선택시 임베딩할 청크 수
    'k': 15  # auto 선택시 선택할 문서 수
}

print("📋 프롬프트 튜닝 설정:")
for key, value in TUNING_CONFIG.items():
    print(f"   {key}: {value}")

print("\n💡 설정 설명:")
print("   • domain: 농업 특화 프롬프트 생성을 위한 도메인 지정")
print("   • selection_method='auto': 대표적인 샘플을 자동으로 선택")
print("   • discover_entity_types: 토마토, 키토산 등 농업 용어 자동 발견")
print("   • language='Korean': 한국어 텍스트 처리에 최적화")

print("\n" + "="*60)

In [ ]:
# Auto Prompt Tuning 실행
print("🚀 Auto Prompt Tuning 실행...")
print("⏰ 예상 소요 시간: 3-10분 (API 응답 속도에 따라 변동)")
print("\n" + "="*60)

# 프롬프트 튜닝 명령어 구성
prompt_tune_cmd = [
    sys.executable, "-m", "graphrag", "prompt-tune",
    "--root", ".",
    "--domain", TUNING_CONFIG['domain'],
    "--language", TUNING_CONFIG['language'],
    "--selection-method", TUNING_CONFIG['selection_method'],
    "--limit", str(TUNING_CONFIG['limit']),
    "--max-tokens", str(TUNING_CONFIG['max_tokens']),
    "--chunk-size", str(TUNING_CONFIG['chunk_size']),
    "--min-examples-required", str(TUNING_CONFIG['min_examples_required']),
    "--n-subset-max", str(TUNING_CONFIG['n_subset_max']),
    "--k", str(TUNING_CONFIG['k'])
]

if TUNING_CONFIG['discover_entity_types']:
    prompt_tune_cmd.append("--discover-entity-types")

print("💻 실행 명령어:")
print(f"   {' '.join(prompt_tune_cmd)}")
print("\n🏃 실행 중...")

# 진행 상황 추적
start_time = time.time()
process_completed = False

try:
    # 프로세스 실행
    process = subprocess.Popen(
        prompt_tune_cmd,
        cwd=WORK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        universal_newlines=True
    )
    
    # 실시간 출력 표시
    for line in process.stdout:
        if line.strip():
            # 주요 진행 사항만 표시
            if any(keyword in line.lower() for keyword in ['loading', 'creating', 'generating', 'done', 'complete']):
                print(f"   → {line.strip()}")
    
    # 프로세스 종료 대기
    process.wait(timeout=600)  # 10분 타임아웃
    
    if process.returncode == 0:
        process_completed = True
        print("\n✅ Auto Prompt Tuning 완료!")
    else:
        print("\n⚠️ 프로세스가 오류와 함께 종료됨")
        stderr_output = process.stderr.read()
        if stderr_output:
            print(f"오류 메시지: {stderr_output[:500]}...")
            
except subprocess.TimeoutExpired:
    print("\n⏱️ 10분 타임아웃 - 프로세스가 너무 오래 걸립니다.")
    process.kill()
    print("💡 기본 프롬프트를 사용하여 진행할 수 있습니다.")
except Exception as e:
    print(f"\n❌ 실행 중 오류: {e}")

# 실행 시간 표시
elapsed_time = time.time() - start_time
print(f"\n⏱️ 소요 시간: {elapsed_time//60:.0f}분 {elapsed_time%60:.0f}초")
print("\n" + "="*60)

In [ ]:
# 생성된 프롬프트 확인
print("🔍 생성된 프롬프트 파일 확인...")

prompts_dir = os.path.join(WORK_DIR, "prompts")

if os.path.exists(prompts_dir):
    prompt_files = [f for f in os.listdir(prompts_dir) if f.endswith('.txt')]
    
    if prompt_files:
        print(f"\n✅ {len(prompt_files)}개의 프롬프트 파일 생성됨:")
        
        # 주요 프롬프트 파일들
        key_prompts = {
            'entity_extraction.txt': '엔티티 추출',
            'extract_graph.txt': '그래프 추출',
            'summarize_descriptions.txt': '설명 요약',
            'community_report.txt': '커뮤니티 리포트',
            'extract_claims.txt': '주장 추출'
        }
        
        for filename in sorted(prompt_files):
            filepath = os.path.join(prompts_dir, filename)
            size = os.path.getsize(filepath)
            description = key_prompts.get(filename, '기타')
            
            print(f"\n📄 {filename} ({description}):")
            print(f"   크기: {size:,} bytes")
            
            # 프롬프트 내용 미리보기 (처음 200자)
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
                
                # 농업 관련 키워드 확인
                agri_keywords = ['agriculture', '농업', 'farming', '재배', 'crop', '작물', 
                               'tomato', '토마토', 'chitosan', '키토산', 'disease', '병해']
                found_keywords = [kw for kw in agri_keywords if kw.lower() in content.lower()]
                
                if found_keywords:
                    print(f"   ✅ 농업 도메인 특화 확인: {', '.join(found_keywords[:3])}")
                else:
                    print(f"   ⚠️ 농업 키워드 미발견 (일반 프롬프트일 수 있음)")
                
                # 내용 미리보기
                preview = content[:200].replace('\n', ' ')
                print(f"   미리보기: {preview}...")
        
        # settings.yaml 업데이트 안내
        print("\n💡 프롬프트 적용 방법:")
        print("   settings.yaml에 다음 설정이 자동으로 추가됩니다:")
        print("   - entity_extraction.prompt: prompts/entity_extraction.txt")
        print("   - summarize_descriptions.prompt: prompts/summarize_descriptions.txt")
        print("   - community_reports.prompt: prompts/community_report.txt")
        
    else:
        print("❌ 프롬프트 파일이 생성되지 않았습니다.")
        print("💡 기본 프롬프트를 사용하여 진행합니다.")
else:
    print("❌ prompts 디렉토리가 생성되지 않았습니다.")
    print("💡 기본 프롬프트를 사용하여 진행합니다.")

## 5. GraphRAG 인덱싱 실행

In [ ]:
# 인덱싱 전 최종 확인
print("🔍 인덱싱 전 최종 확인...")
print("\n" + "="*60)

# 1. 입력 파일 확인
input_files = os.listdir(input_dir)
print(f"✅ 입력 파일: {len(input_files)}개")

# 2. 설정 확인
print("\n✅ 주요 설정:")
with open(settings_file, 'r', encoding='utf-8') as f:
    current_settings = yaml.safe_load(f)
    
    # 모델 설정 확인
    if 'models' in current_settings:
        chat_model = current_settings['models'].get('default_chat_model', {})
        print(f"   채팅 모델: {chat_model.get('model', 'Unknown')}")
        print(f"   API 키 설정: {'✅' if chat_model.get('api_key') else '❌'}")
    
    # 청킹 설정 확인
    if 'chunks' in current_settings:
        chunks = current_settings['chunks']
        print(f"   청크 크기: {chunks.get('size', 'Unknown')} 토큰")
        print(f"   청크 전략: {chunks.get('strategy', 'Unknown')}")
    
    # 프롬프트 설정 확인
    custom_prompts = 0
    for section in ['entity_extraction', 'summarize_descriptions', 'community_reports']:
        if section in current_settings and 'prompt' in current_settings[section]:
            custom_prompts += 1
    
    print(f"   커스텀 프롬프트: {custom_prompts}개")

# 3. 환경 변수 확인
print("\n✅ 환경 변수:")
print(f"   OPENAI_API_KEY: {'설정됨' if 'OPENAI_API_KEY' in os.environ else '미설정'}")
print(f"   GRAPHRAG_API_KEY: {'설정됨' if 'GRAPHRAG_API_KEY' in os.environ else '미설정'}")

print("\n" + "="*60)
print("\n🚀 인덱싱을 시작할 준비가 되었습니다!")
print("\n⚠️ 주의사항:")
print("   • 전체 처리에 5-20분이 소요될 수 있습니다.")
print("   • API 호출로 인한 비용이 발생합니다.")
print("   • 네트워크 연결이 안정적이어야 합니다.")

In [ ]:
# GraphRAG 인덱싱 실행
print("🏗️ GraphRAG 지식그래프 생성 시작...")
print(f"📁 작업 디렉토리: {WORK_DIR}")
print("\n" + "="*60)

# 인덱싱 명령어
index_cmd = [
    sys.executable, "-m", "graphrag", "index",
    "--root", ".",
    "--verbose"  # 상세 로그 출력
]

print("💻 실행 명령어:")
print(f"   {' '.join(index_cmd)}")
print("\n🏃 인덱싱 진행 중...")

# 출력 디렉토리
output_dir = os.path.join(WORK_DIR, "output")

# 진행 상황 모니터링 함수
def monitor_indexing():
    """인덱싱 진행 상황을 실시간으로 모니터링"""
    expected_files = {
        'entities.parquet': '엔티티',
        'relationships.parquet': '관계',
        'communities.parquet': '커뮤니티',
        'community_reports.parquet': '커뮤니티 리포트',
        'text_units.parquet': '텍스트 유닛',
        'documents.parquet': '문서',
        'graph.graphml': '그래프 구조'
    }
    
    created_files = set()
    
    while not stop_monitoring:
        if os.path.exists(output_dir):
            for filename, description in expected_files.items():
                filepath = os.path.join(output_dir, filename)
                
                if os.path.exists(filepath) and filename not in created_files:
                    created_files.add(filename)
                    size = os.path.getsize(filepath)
                    timestamp = datetime.now().strftime("%H:%M:%S")
                    print(f"   [{timestamp}] ✅ {description} 생성 완료 ({size:,} bytes)")
        
        time.sleep(3)

# 모니터링 시작
stop_monitoring = False
monitor_thread = threading.Thread(target=monitor_indexing)
monitor_thread.start()

# 인덱싱 실행
start_time = time.time()

try:
    # 프로세스 실행
    process = subprocess.Popen(
        index_cmd,
        cwd=WORK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    
    # 주요 진행 사항 표시
    for line in process.stdout:
        if line.strip():
            # 워크플로우 진행 상황 표시
            if any(keyword in line for keyword in ['workflow', 'executing', 'completed', 'error']):
                print(f"   → {line.strip()}")
    
    # 프로세스 종료 대기
    process.wait()
    
    if process.returncode == 0:
        print("\n🎉 인덱싱 성공적으로 완료!")
    else:
        print("\n⚠️ 인덱싱이 오류와 함께 종료됨")
        stderr_output = process.stderr.read()
        if stderr_output:
            print(f"오류 메시지: {stderr_output[:1000]}...")
            
except Exception as e:
    print(f"\n❌ 인덱싱 중 오류 발생: {e}")

finally:
    # 모니터링 중지
    stop_monitoring = True
    monitor_thread.join()

# 실행 시간 표시
elapsed_time = time.time() - start_time
print(f"\n⏱️ 총 소요 시간: {elapsed_time//60:.0f}분 {elapsed_time%60:.0f}초")
print("\n" + "="*60)

## 6. 생성된 지식그래프 분석

In [ ]:
# 생성된 파일 분석
print("📊 생성된 지식그래프 분석...")
print("\n" + "="*60)

output_dir = os.path.join(WORK_DIR, "output")

if os.path.exists(output_dir):
    print(f"📁 출력 디렉토리: {output_dir}\n")
    
    # 주요 파일 분석
    analysis_results = {}
    
    key_files = {
        'entities.parquet': {
            'name': '엔티티',
            'key_columns': ['title', 'type', 'description']
        },
        'relationships.parquet': {
            'name': '관계',
            'key_columns': ['source', 'target', 'description']
        },
        'communities.parquet': {
            'name': '커뮤니티',
            'key_columns': ['title', 'level', 'size']
        },
        'text_units.parquet': {
            'name': '텍스트 유닛',
            'key_columns': ['text', 'n_tokens']
        },
        'documents.parquet': {
            'name': '문서',
            'key_columns': ['title', 'raw_content']
        }
    }
    
    for filename, info in key_files.items():
        filepath = os.path.join(output_dir, filename)
        
        if os.path.exists(filepath):
            try:
                # Parquet 파일 읽기
                df = pd.read_parquet(filepath)
                
                print(f"📈 {info['name']} 분석:")
                print(f"   파일: {filename}")
                print(f"   행 수: {len(df):,}개")
                print(f"   크기: {os.path.getsize(filepath):,} bytes")
                
                # 열 정보
                print(f"   열: {', '.join(df.columns)}")
                
                # 주요 정보 추출
                if filename == 'entities.parquet' and 'title' in df.columns:
                    # 엔티티 타입 분석
                    if 'type' in df.columns and df['type'].notna().any():
                        entity_types = df['type'].value_counts().head(10)
                        print("\n   🏷️ 상위 엔티티 타입:")
                        for etype, count in entity_types.items():
                            print(f"      • {etype}: {count}개")
                    
                    # 주요 엔티티 (농업 관련)
                    agri_keywords = ['토마토', '키토산', '병해', '농업', '재배', '작물']
                    agri_entities = df[df['title'].str.contains('|'.join(agri_keywords), case=False, na=False)]
                    
                    if not agri_entities.empty:
                        print("\n   🌱 농업 관련 주요 엔티티:")
                        for _, entity in agri_entities.head(5).iterrows():
                            title = entity.get('title', 'Unknown')
                            etype = entity.get('type', 'Unknown')
                            print(f"      • {title} ({etype})")
                
                elif filename == 'relationships.parquet':
                    # 관계 통계
                    print(f"\n   🔗 관계 통계:")
                    print(f"      • 총 관계 수: {len(df):,}개")
                    if 'weight' in df.columns:
                        print(f"      • 평균 가중치: {df['weight'].mean():.2f}")
                
                elif filename == 'communities.parquet':
                    # 커뮤니티 분석
                    if 'level' in df.columns:
                        levels = df['level'].value_counts().sort_index()
                        print("\n   🏘️ 커뮤니티 계층:")
                        for level, count in levels.items():
                            print(f"      • 레벨 {level}: {count}개")
                
                print("\n" + "-"*50 + "\n")
                
                # 결과 저장
                analysis_results[filename] = {
                    'rows': len(df),
                    'size': os.path.getsize(filepath),
                    'columns': list(df.columns)
                }
                
            except Exception as e:
                print(f"   ❌ {filename} 읽기 실패: {e}\n")
        else:
            print(f"   ❌ {filename} 파일 없음\n")
    
    # 전체 요약
    print("📊 전체 요약:")
    total_files = len([f for f in analysis_results if analysis_results[f]['rows'] > 0])
    total_size = sum(r['size'] for r in analysis_results.values())
    
    print(f"   • 생성된 파일: {total_files}개")
    print(f"   • 전체 크기: {total_size:,} bytes ({total_size/1024/1024:.1f} MB)")
    
    if 'entities.parquet' in analysis_results:
        print(f"   • 총 엔티티: {analysis_results['entities.parquet']['rows']:,}개")
    if 'relationships.parquet' in analysis_results:
        print(f"   • 총 관계: {analysis_results['relationships.parquet']['rows']:,}개")
    
    print("\n✅ 지식그래프가 성공적으로 생성되었습니다!")
    
else:
    print("❌ 출력 디렉토리를 찾을 수 없습니다.")
    print("   인덱싱이 제대로 실행되지 않았을 수 있습니다.")

## 7. 쿼리 테스트

In [ ]:
# 샘플 쿼리 준비
print("🔍 GraphRAG 쿼리 기능 테스트")
print("\n" + "="*60)

# 농업 관련 테스트 쿼리
test_queries = [
    {
        'query': "토마토 잎곰팡이병의 주요 방제 방법은 무엇인가요?",
        'type': 'local',
        'description': '특정 병해 방제법'
    },
    {
        'query': "키토산이 농업에서 사용되는 주요 용도와 효과를 설명해주세요.",
        'type': 'local',
        'description': '특정 물질의 용도'
    },
    {
        'query': "전체 문서에서 다루는 농업 병해충 방제의 주요 접근법들을 요약해주세요.",
        'type': 'global',
        'description': '전체적인 패턴 파악'
    },
    {
        'query': "친환경 농업 방법들과 그 효과에 대해 종합적으로 설명해주세요.",
        'type': 'global',
        'description': '주제별 종합 분석'
    }
]

print("📋 테스트 쿼리 목록:")
for i, q in enumerate(test_queries, 1):
    print(f"\n{i}. [{q['type'].upper()}] {q['description']}")
    print(f"   질문: {q['query']}")

print("\n" + "="*60)

In [ ]:
# 쿼리 실행 함수
def run_graphrag_query(query_text, method='local', verbose=False):
    """
    GraphRAG 쿼리 실행
    
    Args:
        query_text: 질문 텍스트
        method: 'local' 또는 'global'
        verbose: 상세 출력 여부
    
    Returns:
        응답 텍스트 또는 None
    """
    query_cmd = [
        sys.executable, "-m", "graphrag", "query",
        "--root", ".",
        "--method", method,
        "--query", query_text  # 수정: --query 옵션 추가
    ]
    
    if verbose:
        query_cmd.append("--verbose")
    
    try:
        result = subprocess.run(
            query_cmd,
            cwd=WORK_DIR,
            capture_output=True,
            text=True,
            timeout=120  # 2분 타임아웃
        )
        
        if result.returncode == 0:
            return result.stdout.strip()
        else:
            return f"오류 발생: {result.stderr}"
            
    except subprocess.TimeoutExpired:
        return "타임아웃: 응답 시간 초과"
    except Exception as e:
        return f"실행 오류: {e}"

# 쿼리 실행
print("🏃 쿼리 실행 중...\n")

for i, test_query in enumerate(test_queries, 1):
    print(f"\n{'='*60}")
    print(f"쿼리 {i}/{len(test_queries)}: {test_query['description']}")
    print(f"방법: {test_query['type'].upper()} Search")
    print(f"질문: {test_query['query']}")
    print("-"*60)
    
    # 쿼리 실행
    print("\n응답을 생성하는 중...")
    response = run_graphrag_query(
        test_query['query'], 
        method=test_query['type']
    )
    
    if response:
        # 응답 포맷팅
        if len(response) > 800:
            # 긴 응답은 요약
            print("\n📝 응답 (요약):")
            print(response[:800] + "...\n[응답이 너무 길어 일부만 표시]")
        else:
            print("\n📝 응답:")
            print(response)
    else:
        print("\n❌ 응답 생성 실패")
    
    print("\n" + "="*60)
    
    # 잠시 대기 (API 제한 고려)
    time.sleep(2)

print("\n✅ 모든 쿼리 테스트 완료!")

## 8. 기존 노트북과의 차이점 분석

In [ ]:
# 차이점 분석 문서 생성
print("📝 기존 노트북과의 차이점 분석")
print("\n" + "="*60)

differences = """
# GraphRAG 구현 비교: 기존 vs 공식 문서 기반

## 1. 프로젝트 구조

### 기존 노트북 (graphrag.ipynb)
- 작업 위치: GraphRAG 소스 디렉토리 내부
- 파일 구조: 소스코드와 작업 파일 혼재
- 설정 방식: 직접적인 파일 수정

### 새 노트북 (graphrag_official.ipynb) ✨
- 작업 위치: 별도 workspace 디렉토리 사용
- 파일 구조: 소스코드와 작업공간 명확히 분리
- 설정 방식: 공식 CLI 명령어 사용
- 장점: 더 깔끔하고 관리하기 쉬운 구조

## 2. 초기화 과정

### 기존 방식
```python
# 수동으로 설정 파일 생성
settings = {...}
yaml.dump(settings, open('settings.yaml', 'w'))
```

### 새 방식 ✨
```python
# 공식 CLI 명령어 사용
subprocess.run(["python", "-m", "graphrag", "init", "--root", "."])
```
- 장점: 표준화된 프로젝트 구조 자동 생성

## 3. API 키 처리

### 기존 방식
- 환경 변수 의존성이 높음
- API 키 직접 삽입 방식 혼재

### 새 방식 ✨
- .env 파일 생성으로 표준화
- OpenAI API 직접 테스트로 연결 검증
- 더 안정적인 API 키 관리

## 4. 프롬프트 튜닝

### 기존 방식
- 기본 파라미터 사용
- 타임아웃 처리 미흡

### 새 방식 ✨
- 한국 농업 도메인 특화 파라미터
- selection_method='auto'로 대표 샘플 자동 선택
- 실시간 진행 상황 표시
- 타임아웃 및 에러 처리 강화

## 5. 인덱싱 프로세스

### 기존 방식
- 단순 명령 실행
- 진행 상황 파악 어려움

### 새 방식 ✨
- 실시간 파일 생성 모니터링
- 워크플로우 단계별 진행 표시
- 생성된 파일 즉시 분석

## 6. 결과 분석

### 기존 방식
- 기본적인 파일 크기 확인

### 새 방식 ✨
- Parquet 파일 내용 상세 분석
- 농업 관련 엔티티 필터링
- 커뮤니티 계층 구조 분석
- 관계 통계 표시

## 7. 쿼리 테스트

### 기존 방식
- 간단한 샘플 쿼리

### 새 방식 ✨
- 농업 도메인 특화 쿼리 세트
- Local/Global Search 구분
- 쿼리 유형별 설명 추가
- 응답 품질 평가

## 8. 에러 처리 및 사용성

### 기존 방식
- 기본적인 에러 메시지

### 새 방식 ✨
- 단계별 상세한 진행 상황 표시
- 문제 발생시 해결 방법 안내
- 각 단계별 성공/실패 명확히 표시
- 한국어 설명과 이모지로 가독성 향상

## 결론

새 노트북(graphrag_official.ipynb)은 Microsoft GraphRAG의 공식 문서와 
Best Practice를 충실히 따르면서도, 한국 농업 데이터의 특성을 고려한 
최적화를 적용했습니다. 특히 실무에서 발생할 수 있는 다양한 에러 상황에 
대한 처리와 사용자 친화적인 인터페이스를 제공합니다.
"""

# 차이점 문서 저장
diff_file = os.path.join(PROJECT_ROOT, "kg_gen/graphrag_comparison.md")
with open(diff_file, 'w', encoding='utf-8') as f:
    f.write(differences)

print(f"✅ 차이점 분석 문서 저장: {diff_file}")

# 주요 차이점 요약 출력
print("\n📊 주요 개선사항 요약:")
improvements = [
    "워크스페이스 분리로 더 깔끔한 프로젝트 구조",
    "공식 CLI 명령어 사용으로 표준화",
    "한국 농업 도메인 특화 프롬프트 튜닝",
    "실시간 진행 상황 모니터링",
    "상세한 결과 분석 및 농업 엔티티 필터링",
    "강화된 에러 처리 및 사용자 가이드"
]

for i, improvement in enumerate(improvements, 1):
    print(f"   {i}. ✨ {improvement}")

print("\n" + "="*60)

## 9. 마무리 및 다음 단계

### 🎉 완료된 작업:
1. **GraphRAG 공식 파이프라인 구축**: Microsoft 권장 방식 준수
2. **한국 농업 특화 프롬프트**: Auto Prompt Tuning으로 도메인 적응
3. **지식그래프 생성**: 엔티티, 관계, 커뮤니티 구조 완성
4. **쿼리 기능 검증**: Local/Global Search 테스트 완료

### 📁 생성된 파일들:
- `graphrag_workspace/`: 전체 작업 공간
- `graphrag_workspace/output/`: 생성된 지식그래프 파일들
- `graphrag_workspace/prompts/`: 농업 도메인 특화 프롬프트
- `graphrag_comparison.md`: 기존 구현과의 차이점 문서

### 🚀 다음 단계:
1. **성능 평가**: kg-gen과의 비교 분석
2. **쿼리 최적화**: 더 복잡한 농업 관련 질문 테스트
3. **확장**: 더 많은 농업 문서 추가
4. **통합**: RAG 시스템에 GraphRAG 통합

### 💡 활용 팁:
- 새로운 문서 추가시: `input/` 디렉토리에 텍스트 파일 추가 후 재인덱싱
- 프롬프트 수정: `prompts/` 디렉토리의 파일 직접 편집 가능
- 쿼리 실행: `graphrag query --root . --method [local|global] "질문"`